In [19]:
import numpy as np
import os
from PIL import Image, ImageOps
from IPython.display import display


In [9]:
# File paths to different assets
file_enemy = "enemy"
file_env = "environment"

In [ ]:
def load_environment_images(directory_path):
    # Load all PNG images from the specified directory
    for file_name in os.listdir(directory_path):
        if file_name.endswith('.png'):
            img = Image.open(os.path.join(directory_path, file_name))
            # Resize the image to 256x256
            img_resized = img.resize((256, 256))
            # Display the resized image
            display(img_resized)

load_environment_images(file_env)

In [16]:
def augment_image(img):
    # Flip the image horizontally
    img_flipped = ImageOps.mirror(img)
    
    # Mirror left half to right
    left_half = img.crop((0, 0, img.width // 2, img.height))
    left_mirrored = ImageOps.mirror(left_half)
    img_left_mirror = Image.new('RGB', (img.width, img.height))
    img_left_mirror.paste(left_half, (0, 0))
    img_left_mirror.paste(left_mirrored, (img.width // 2, 0))
    
    # Mirror right half to left
    right_half = img.crop((img.width // 2, 0, img.width, img.height))
    right_mirrored = ImageOps.mirror(right_half)
    img_right_mirror = Image.new('RGB', (img.width, img.height))
    img_right_mirror.paste(right_mirrored, (0, 0))
    img_right_mirror.paste(right_half, (img.width // 2, 0))
    
    # Copy 1/4 of the image to create 4 new images
    quarter_width = img.width // 2
    quarter_height = img.height // 2
    quarters = [
        img.crop((0, 0, quarter_width, quarter_height)),
        img.crop((quarter_width, 0, img.width, quarter_height)),
        img.crop((0, quarter_height, quarter_width, img.height)),
        img.crop((quarter_width, quarter_height, img.width, img.height))
    ]
    img_quarters = []
    for quarter in quarters:
        new_img = Image.new('RGB', (img.width, img.height))
        new_img.paste(quarter, (0, 0))
        new_img.paste(quarter, (quarter_width, 0))
        new_img.paste(quarter, (0, quarter_height))
        new_img.paste(quarter, (quarter_width, quarter_height))
        img_quarters.append(new_img)
    
    return [img_flipped, img_left_mirror, img_right_mirror] + img_quarters

In [25]:
def load_and_augment_images(directory_path, output_directory):
    # Create the output directory if it doesn't exist
    if not os.path.exists(output_directory):
        os.makedirs(output_directory)
    
    # Load all PNG images from the specified directory
    for file_name in os.listdir(directory_path):
        if file_name.endswith('.png'):
            img = Image.open(os.path.join(directory_path, file_name))
            # Resize the image to 256x256
            img_resized = img.resize((256, 256))
            # Display the original image
            #display(img_resized)
            # Perform augmentations
            augmented_images = augment_image(img_resized)
            # Save the original and augmented images
            base_name = os.path.splitext(file_name)[0]
            img_resized.save(os.path.join(output_directory, f"{base_name}_original.png"))
            for i, aug_img in enumerate(augmented_images):
                aug_img.save(os.path.join(output_directory, f"{base_name}_aug_{i}.png"))
            #break  # Remove this line to process all images in the directory


In [26]:
load_and_augment_images(file_env, 'env_processed')